# CATE Spatial Maps Notebook

本笔记用于**单独绘制 CATE 空间地图**，基于 `scripts/14d_causal_forest.R` 生成的 `output/14_causal/d_cate/cate_result_*.csv` 结果。

- 请先在 R 中运行 `14d_causal_forest.R`，确保已经生成 CATE 结果 CSV 文件；
- 然后在项目根目录 `E:/CausalSDMs` 下启动 Jupyter，打开本笔记；
- 通过下方代码单元选择待展示的处理变量 (treatment variable)，绘制高分辨率（1200 dpi）的英文标注 CATE 空间图。

代码单元内全部使用中文注释，图内文字和坐标轴等均为英文。

In [ ]:
## 导入依赖库，并设置绘图基础样式（Arial 字体 + 1200 dpi 保存）

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 使用无衬线 Arial 字体，确保与论文配图风格一致
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False  # 避免负号显示为方块

# 工程根目录（默认按当前文件位置推断；如不正确可手动修改）
NB_PATH = Path(__file__).resolve()
PROJECT_ROOT = NB_PATH.parents[3]  # 对应 E:/CausalSDMs

print("Project root:", PROJECT_ROOT)

# CATE 结果与出图目录
CATE_DIR = PROJECT_ROOT / "output" / "d_cate"
FIG_DIR = PROJECT_ROOT / "figures" / "d_cate"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("CATE csv dir:", CATE_DIR)
print("Figure output dir:", FIG_DIR)

# 列出目前可用的 CATE 结果文件，便于交互选择
cate_files = sorted(CATE_DIR.glob("cate_result_*.csv"))
if not cate_files:
    print("[警告] 未在 output/14_causal/d_cate 下找到 cate_result_*.csv，请先运行 14d_causal_forest.R。")
else:
    print("可用的 CATE 结果文件：")
    for f in cate_files:
        print(" -", f.name)

In [ ]:
## 定义绘制单个 CATE 空间地图的函数

def plot_cate_map(csv_path, vlim=None, save_fig=True, dpi=1200):
    """基于单个 CATE 结果 CSV 绘制空间分布图。

    参数
    ------
    csv_path : Path 或 str
        `cate_result_*.csv` 文件路径，须包含列 `lon`, `lat`, `cate`, `is_valid`。
    vlim : float 或 None
        颜色范围上界；若为 None，则按 0.9 * max(|CATE|) 自动设置，对称 [-vlim, vlim]。
    save_fig : bool
        是否将结果保存为 PNG + SVG 文件。
    dpi : int
        保存图片的分辨率（论文建议 1200 dpi）。
    """

    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    if not {"lon", "lat", "cate"}.issubset(df.columns):
        raise ValueError("CSV 文件中必须至少包含列: lon, lat, cate")

    # 只使用有效样本（与 14d_causal_forest.R 保持一致）
    if "is_valid" in df.columns:
        df_valid = df[df["is_valid"] == True].copy()
    else:
        df_valid = df.copy()

    if df_valid.empty:
        raise ValueError("有效样本为空，无法绘制 CATE 地图。")

    # 变量与标题标签（与 14d 脚本约定保持一致）
    treat_var = df_valid["treatment_var"].iloc[0] if "treatment_var" in df_valid.columns else "unknown_var"
    var_label = (treat_var
                 .replace("flow_length", "Flow Length")
                 .replace("lc_wavg_09", "Urbanization"))

    # 颜色范围：默认按 0.9 * max(|CATE|) 对称设置
    if vlim is None:
        vlim = 0.9 * np.nanmax(np.abs(df_valid["cate"].values))
    if vlim <= 0:
        vlim = 1.0

    # 开始绘图：单幅散点图（经纬度坐标）
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

    sc = ax.scatter(
        df_valid["lon"], df_valid["lat"],
        c=df_valid["cate"],
        s=4,
        cmap="coolwarm",
        vmin=-vlim,
        vmax=vlim,
        alpha=0.8,
        edgecolors="none",
    )

    cbar = fig.colorbar(sc, ax=ax, shrink=0.85)
    cbar.set_label("CATE", fontsize=10)

    ax.set_xlabel("Longitude", fontsize=10)
    ax.set_ylabel("Latitude", fontsize=10)
    ax.set_title(
        f"Spatial CATE map: {var_label}",
        fontsize=12,
    )
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linewidth=0.3, color="lightgrey", alpha=0.7)

    fig.tight_layout()

    # 保存为高分辨率图片（1200 dpi），PNG + SVG 双格式
    if save_fig:
        stem = csv_path.stem.replace("cate_result_", "cate_map_")
        out_png = FIG_DIR / f"{stem}.png"
        out_svg = FIG_DIR / f"{stem}.svg"
        fig.savefig(out_png, dpi=dpi, bbox_inches="tight", facecolor="white")
        fig.savefig(out_svg, dpi=dpi, bbox_inches="tight", facecolor="white")
        print(f"Saved: {out_png}")
        print(f"Saved: {out_svg}")

    return fig, ax

In [ ]:
## 示例：选择一个 CATE 结果文件并绘制空间地图

# 在此处手动指定要绘制的文件名（需与 output/14_causal/d_cate 中存在的文件对应）
# 例如: "cate_result_flow_length.csv" 或 "cate_result_lc_wavg_09.csv"
example_name = "cate_result_flow_length.csv"  # TODO: 根据实际文件名修改

example_path = CATE_DIR / example_name
if not example_path.exists():
    print(f"[提示] 文件不存在: {example_path}, 请先检查文件名或运行 14d_causal_forest.R。")
else:
    _fig, _ax = plot_cate_map(example_path, vlim=None, save_fig=True, dpi=1200)
    plt.show()